<a href="https://colab.research.google.com/github/vanderbarbosa/mestrado/blob/main/02_Web_Scraping_Noticias_PETR4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PASSO 1: PREPARAÇÃO DE BIBLIOTECAS E INTEGRAÇÃO
# ==============================================================================
# Instalando a biblioteca GoogleNews para buscas históricas automatizadas
!pip install GoogleNews --quiet

from google.colab import drive
import os
import pandas as pd
from GoogleNews import GoogleNews
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta

drive.mount('/content/drive')
caminho_base = '/content/drive/MyDrive/Mestrado_PETR4/'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 24.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# ==============================================================================
# PASSO 2: CONFIGURAÇÃO DO ROBÔ DE SCRAPING HISTÓRICO
# ==============================================================================
termo_busca = "Petrobras"
print(f"Iniciando o Robô Histórico de Coleta de Notícias para o termo: '{termo_busca}'")

# Definindo nossa janela de pesquisa em blocos mensais (para não sobrecarregar a busca)
data_inicial = datetime.strptime("2018-01-01", "%Y-%m-%d")
data_final = datetime.strptime("2025-12-31", "%Y-%m-%d")
data_atual_loop = data_inicial

lista_noticias_historicas = []

Iniciando o Robô Histórico de Coleta de Notícias para o termo: 'Petrobras'


In [3]:
# ==============================================================================
# PASSO 3: LAÇO DE REPETIÇÃO (ITERANDO DE 2018 A 2025)
# ==============================================================================
while data_atual_loop < data_final:
    # Definindo o início e o fim do mês atual no loop
    mes_inicio_str = data_atual_loop.strftime("%m/%d/%Y")
    data_proximo_mes = data_atual_loop + relativedelta(months=1)
    mes_fim_str = data_proximo_mes.strftime("%m/%d/%Y")

    print(f"Minerando notícias do período: {mes_inicio_str} a {mes_fim_str}...")

    # Configurando o motor de busca para o idioma português e a janela de datas
    googlenews = GoogleNews(lang='pt', region='BR')
    googlenews.set_time_range(mes_inicio_str, mes_fim_str)

    try:
        googlenews.search(termo_busca)

        # Paginando os resultados (capturando até as 3 primeiras páginas de cada mês para amostra)
        for pagina in range(1, 4):
            googlenews.get_page(pagina)
            resultados = googlenews.results()

            for noticia in resultados:
                # Filtrando para evitar notícias duplicadas
                if noticia['title'] not in [n['Titulo'] for n in lista_noticias_historicas]:
                    lista_noticias_historicas.append({
                        'Data_Publicacao': noticia.get('datetime', mes_inicio_str), # Timestamp aproximado
                        'Ativo': 'PETR4',
                        'Titulo': noticia['title'],
                        'Resumo': noticia.get('desc', noticia['title']), # O GoogleNews traz um resumo (snippet)
                        'Fonte': noticia.get('media', 'Desconhecida')
                    })

            # Pausa de cortesia (Politeness) para evitar bloqueio pelo servidor
            time.sleep(2)

    except Exception as e:
        print(f"Erro na extração do mês {mes_inicio_str}: {e}")

    googlenews.clear()

    # Avança o loop para o próximo mês
    data_atual_loop = data_proximo_mes

Minerando notícias do período: 01/01/2018 a 02/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 02/01/2018 a 03/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 03/01/2018 a 04/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 04/01/2018 a 05/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 05/01/2018 a 06/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 06/01/2018 a 07/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 07/01/2018 a 08/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 08/01/2018 a 09/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 09/01/2018 a 10/01/2018...
'NoneType' object has no attribute 'get'
Minerando notícias do período: 10/01/2018 a 11/01/2018...
'NoneType' object has no attribute 'get'
Minerando 

In [4]:
# ==============================================================================
# PASSO 4: ESTRUTURAÇÃO E SALVAMENTO DO CORPUS DE BIG DATA
# ==============================================================================
df_noticias_bigdata = pd.DataFrame(lista_noticias_historicas)

# Renomeando a coluna Data para o padrão que nosso script do BERTimbau espera
df_noticias_bigdata.rename(columns={'Data_Publicacao': 'Data_Coleta'}, inplace=True)

caminho_arquivo_textual = caminho_base + "base_textual_petr4_2018_2025.csv"
df_noticias_bigdata.to_csv(caminho_arquivo_textual, index=False, encoding='utf-8')

print("\n" + "="*50)
print(f"EXTRAÇÃO FINALIZADA COM SUCESSO!")
print(f"Total de Notícias Históricas Capturadas: {len(df_noticias_bigdata)}")
print(f"Arquivo salvo em: {caminho_arquivo_textual}")
print("="*50)


EXTRAÇÃO FINALIZADA COM SUCESSO!
Total de Notícias Históricas Capturadas: 9080
Arquivo salvo em: /content/drive/MyDrive/Mestrado_PETR4/base_textual_petr4_2018_2025.csv
